# 02 — Build Analytics Base Table (ABT)

**Project:** Gyakriti — Olist Brazilian E-Commerce Analytics  
**Grain:** One row = One Order Item  
**Notebook purpose:** Assemble a single, analysis-ready flat table from the raw Olist source tables.  
The ABT is the single source of truth for all downstream analysis and dashboards.

---

### Merge sequence

| Step | Table | Join key | Type |
|------|-------|----------|------|
| Base | `order_items` | — | — |
| 1 | `orders` | `order_id` | left |
| 2 | `customers` | `customer_id` | left |
| 3 | `products` | `product_id` | left |
| 4 | `category_translation` | `product_category_name` | left |
| 5 | `sellers` | `seller_id` | left |
| 6 | `reviews` (deduped) | `order_id` | left |
| 7 | `payments` (aggregated) | `order_id` | left |

> **Rule:** After every merge, the row count must equal the base count of `order_items` (112 650).  
> Any unexpected row increase triggers an assertion error before continuing.

---
## 0 — Environment Setup

In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Make src/ importable from this notebook
PROJECT_ROOT = Path("/Users/vishwashpatidar/Project Gyakriti/Gyakriti")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from audit import dataframe_overview
from validation import validate_primary_key, validate_foreign_key, validation_summary

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

---
## 1 — Load Raw Data

All source tables are loaded once here and kept untouched throughout the notebook.  
Timestamp columns are parsed at load time to avoid repeated `pd.to_datetime()` calls downstream.

In [3]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

TIMESTAMP_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

order_items  = pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv")
orders       = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv",
                           parse_dates=TIMESTAMP_COLS)
customers    = pd.read_csv(RAW_DATA_PATH / "olist_customers_dataset.csv")
products     = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")
prod_cat     = pd.read_csv(RAW_DATA_PATH / "product_category_name_translation.csv")
sellers      = pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv")
reviews      = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")
payments     = pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv")

In [4]:
# Record the canonical row count — every merge must preserve this exactly.
EXPECTED_ROWS = len(order_items)

dataset_shapes = pd.DataFrame(
    {
        "rows": [df.shape[0] for df in [order_items, orders, customers,
                                         products, prod_cat, sellers,
                                         reviews, payments]],
        "columns": [df.shape[1] for df in [order_items, orders, customers,
                                             products, prod_cat, sellers,
                                             reviews, payments]],
    },
    index=["order_items", "orders", "customers", "products",
           "prod_cat", "sellers", "reviews", "payments"],
)

print(f"Base grain: {EXPECTED_ROWS:,} order item rows\n")
dataset_shapes

Base grain: 112,650 order item rows



,rows,columns
order_items,112650,7
orders,99441,8
customers,99441,5
products,32951,9
prod_cat,71,2
sellers,3095,4
reviews,99224,7
payments,103886,5


---
## 2 — Pre-Merge Validation

Before building the ABT, we confirm the key integrity of every source table.  
A failed check here signals a data quality problem that must be resolved upstream.

In [5]:
# --- Primary key checks ---
pk_order_items = validate_primary_key(order_items, ["order_id", "order_item_id"])
pk_orders      = validate_primary_key(orders, "order_id")
pk_customers   = validate_primary_key(customers, "customer_id")
pk_products    = validate_primary_key(products, "product_id")
pk_sellers     = validate_primary_key(sellers, "seller_id")

# --- Foreign key checks ---
fk_order_to_orders    = validate_foreign_key(order_items, "order_id",
                                             orders, "order_id")
fk_order_to_customer  = validate_foreign_key(orders, "customer_id",
                                             customers, "customer_id")
fk_item_to_product    = validate_foreign_key(order_items, "product_id",
                                             products, "product_id")
fk_item_to_seller     = validate_foreign_key(order_items, "seller_id",
                                             sellers, "seller_id")

summary = validation_summary({
    "PK: order_items (order_id + item_id)": pk_order_items,
    "PK: orders (order_id)": pk_orders,
    "PK: customers (customer_id)": pk_customers,
    "PK: products (product_id)": pk_products,
    "PK: sellers (seller_id)": pk_sellers,
    "FK: order_items → orders": fk_order_to_orders,
    "FK: orders → customers": fk_order_to_customer,
    "FK: order_items → products": fk_item_to_product,
    "FK: order_items → sellers": fk_item_to_seller,
})

summary

,validation,status,issues,notes
0,PK: order_items (order_id + item_id),PASS,0,All checks passed.
1,PK: orders (order_id),PASS,0,All checks passed.
2,PK: customers (customer_id),PASS,0,All checks passed.
3,PK: products (product_id),PASS,0,All checks passed.
4,PK: sellers (seller_id),PASS,0,All checks passed.
5,FK: order_items → orders,PASS,0,All checks passed.
6,FK: orders → customers,PASS,0,All checks passed.
7,FK: order_items → products,PASS,0,All checks passed.
8,FK: order_items → sellers,PASS,0,All checks passed.


---
## 3 — Select Columns from Base Table

We start with `order_items` and keep only the columns needed in the ABT.  
This is the **grain-defining** table — every subsequent merge targets this.

In [6]:
abt = order_items[
    ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"]
].copy()

print(f"Base rows: {len(abt):,}")
abt.head(3)

Base rows: 112,650


,order_id,order_item_id,product_id,seller_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87


---
## 4 — Merge: Orders

`orders` is a 1-to-many table relative to `order_items` (one order can contain multiple items).  
Joining on `order_id` is safe because `order_id` is confirmed unique in `orders`.  
We use a left join to retain every order item even if (in edge cases) no matching order record exists.

In [7]:
ORDERS_COLS = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

abt = abt.merge(
    orders[ORDERS_COLS],
    on="order_id",
    how="left",
)

assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging orders: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After orders merge  → {len(abt):,} rows ✓")

After orders merge  → 112,650 rows ✓


---
## 5 — Merge: Customers

`customer_id` from `orders` links to the `customers` dimension table.  
We keep `customer_unique_id` (the true customer entity) rather than `customer_id`  
(which is a per-order surrogate key).  
City and state capture the delivery geography.

In [8]:
CUSTOMERS_COLS = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
]

abt = abt.merge(
    customers[CUSTOMERS_COLS],
    on="customer_id",
    how="left",
)

# customer_id is no longer needed after bringing in customer_unique_id
abt = abt.drop(columns=["customer_id"])

assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging customers: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After customers merge → {len(abt):,} rows ✓")

After customers merge → 112,650 rows ✓


---
## 6 — Merge: Products

`product_id` is a foreign key in `order_items` pointing to the `products` dimension.  
We carry `product_category_name` (Portuguese) here; the English translation is joined next.  
`product_weight_g` is included as a physical attribute relevant to shipping analysis.

In [9]:
PRODUCTS_COLS = [
    "product_id",
    "product_category_name",
    "product_weight_g",
]

abt = abt.merge(
    products[PRODUCTS_COLS],
    on="product_id",
    how="left",
)

assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging products: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After products merge  → {len(abt):,} rows ✓")

After products merge  → 112,650 rows ✓


---
## 7 — Merge: Category Translation

The `product_category_name` field is in Portuguese in the source data.  
We join the translation table to get the English equivalent.  
This merge is safe: `product_category_name` is unique in `prod_cat` (71 rows, 71 categories).

In [10]:
abt = abt.merge(
    prod_cat[["product_category_name", "product_category_name_english"]],
    on="product_category_name",
    how="left",
)

assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging category translation: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After category merge  → {len(abt):,} rows ✓")

After category merge  → 112,650 rows ✓


---
## 8 — Merge: Sellers

`seller_id` in `order_items` links to the `sellers` dimension.  
We bring in seller geography (city and state) to enable supply-side segmentation.  
`seller_id` is confirmed unique in the `sellers` table, so this is a clean many-to-one join.

In [11]:
SELLERS_COLS = [
    "seller_id",
    "seller_city",
    "seller_state",
]

abt = abt.merge(
    sellers[SELLERS_COLS],
    on="seller_id",
    how="left",
)

assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging sellers: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After sellers merge   → {len(abt):,} rows ✓")

After sellers merge   → 112,650 rows ✓


---
## 9 — Merge: Reviews

**Key consideration:** `reviews` is NOT a clean 1-to-1 table on `order_id`.  
Some orders have multiple reviews (e.g., edited submissions), which would fan-out our grain.  

**Strategy:** Keep only the most recent review per order (latest `review_answer_timestamp`).  
This reduces reviews to a safe 1-per-order structure before joining.

In [12]:
# Parse timestamps so we can sort correctly
reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"], errors="coerce"
)

# Keep the most recent review per order
reviews_deduped = (
    reviews
    .sort_values("review_answer_timestamp", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "review_score"]]
)

print(f"Reviews raw:     {len(reviews):,} rows")
print(f"Reviews deduped: {len(reviews_deduped):,} rows (1 per order)")

Reviews raw:     99,224 rows
Reviews deduped: 98,673 rows (1 per order)


In [13]:
abt = abt.merge(
    reviews_deduped,
    on="order_id",
    how="left",
)

# review_score is nullable — not every order item has a review
assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging reviews: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After reviews merge   → {len(abt):,} rows ✓")
print(f"review_score nulls: {abt['review_score'].isna().sum():,}")

After reviews merge   → 112,650 rows ✓
review_score nulls: 942


---
## 10 — Aggregate & Merge: Payments

**Key consideration:** The `payments` table has one row per payment instrument per order  
(an order can be split across credit card + voucher, for example).  
Merging the raw table directly would violate the order-item grain.

**Strategy:** Aggregate to one row per `order_id` before joining:  
- `total_payment_value` — sum of all payment values for the order  
- `payment_installments` — maximum installments used (most burdensome plan)  
- `primary_payment_type` — the payment type used in the largest single payment  

In [14]:
def get_primary_payment_type(group: pd.DataFrame) -> str:
    """Return the payment type associated with the highest payment value."""
    return group.loc[group["payment_value"].idxmax(), "payment_type"]


payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .apply(
        lambda g: pd.Series({
            "total_payment_value": g["payment_value"].sum(),
            "payment_installments": g["payment_installments"].max(),
            "primary_payment_type": get_primary_payment_type(g),
        }),
        include_groups=False,
    )
    .reset_index(drop=True)
)

print(f"Payments raw:         {len(payments):,} rows")
print(f"Payments aggregated:  {len(payments_agg):,} rows (1 per order)")
payments_agg.head(3)

Payments raw:         103,886 rows
Payments aggregated:  99,440 rows (1 per order)


,order_id,total_payment_value,payment_installments,primary_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card


In [15]:
abt = abt.merge(
    payments_agg,
    on="order_id",
    how="left",
)

assert len(abt) == EXPECTED_ROWS, (
    f"Row count changed after merging payments: expected {EXPECTED_ROWS:,}, got {len(abt):,}"
)
print(f"After payments merge  → {len(abt):,} rows ✓")

After payments merge  → 112,650 rows ✓


---
## 11 — Final Column Ordering

Re-order columns into logical groups for readability:  
**IDs → Order facts → Customer → Product → Seller → Review → Payment**

In [16]:
COLUMN_ORDER = [
    # --- Identifiers ---
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_unique_id",

    # --- Order status & dates ---
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",

    # --- Item financials ---
    "price",
    "freight_value",

    # --- Customer geography ---
    "customer_city",
    "customer_state",

    # --- Product attributes ---
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",

    # --- Seller geography ---
    "seller_city",
    "seller_state",

    # --- Review ---
    "review_score",

    # --- Payment (order-level aggregates) ---
    "total_payment_value",
    "payment_installments",
    "primary_payment_type",
]

abt = abt[COLUMN_ORDER]

print(f"Final ABT shape: {abt.shape[0]:,} rows × {abt.shape[1]} columns")

Final ABT shape: 112,650 rows × 24 columns


---
## 12 — Post-Build Quality Check

Run `audit.dataframe_overview()` for a quick sanity check on the completed ABT,  
then inspect missingness column by column to flag anything unexpected.

In [17]:
dataframe_overview(abt)

,rows,columns,total_cells,missing_cells,missing_pct,duplicate_rows,memory_mb
0,112650,24,2703600,7862,0.29,0,94.79


In [18]:
# Inspect missing value counts across the ABT
# Nulls in date columns are expected (not all orders reach delivery).
# Nulls in review_score indicate orders without a review — also expected.
missing_summary = (
    abt.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda df: (df["missing_count"] / len(abt) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

missing_summary

,missing_count,missing_pct
order_delivered_customer_date,2454,2.18
product_category_name_english,1627,1.44
product_category_name,1603,1.42
order_delivered_carrier_date,1194,1.06
review_score,942,0.84
product_weight_g,18,0.02
order_approved_at,15,0.01
total_payment_value,3,0.00
payment_installments,3,0.00
primary_payment_type,3,0.00


In [19]:
# Confirm dtypes look sensible
abt.dtypes.to_frame("dtype")

,dtype
order_id,str
order_item_id,int64
product_id,str
seller_id,str
customer_unique_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]


In [20]:
# Final look at 5 random rows
abt.sample(5, random_state=42)

,order_id,order_item_id,product_id,seller_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,price,freight_value,customer_city,customer_state,product_category_name,product_category_name_english,product_weight_g,seller_city,seller_state,review_score,total_payment_value,payment_installments,primary_payment_type
107777,f4ee4273538924bda6212f5948e80fde,1,69455f41626a745aea9ee9164cb9eafd,7d13fca15225358621be4086e1eb0964,8cebd890b95f34cd04b08f958c1af21c,delivered,2018-06-12 08:59:54,2018-06-12 09:18:15,2018-06-15 16:36:00,2018-06-25 16:08:40,2018-07-13,180.00,20.45,sao pedro da aldeia,RJ,relogios_presentes,watches_gifts,501.00,ribeirao preto,SP,5.00,200.45,3.00,credit_card
2391,056349f85a73d794119c4286c95a52de,1,af35be35db4ad0dc288b571453337376,d20b021d3efdf267a402c402a48ea64b,8f3dca4306d5a89e4ae2c65c110603a2,delivered,2017-03-03 14:21:58,2017-03-03 14:35:09,2017-03-03 16:08:50,2017-03-17 12:52:17,2017-03-28,10.99,16.05,brasilia,DF,moveis_decoracao,furniture_decor,100.00,ibitinga,SP,5.00,27.04,2.00,credit_card
77829,b124967afcc82ef17ec41020fe2a9136,1,12e6d0f655986ceff00c74658dec97b1,3be634553519fb6536a03e1358e9fdc7,da6ad3471341a3f6801cb11ea9a88b3c,delivered,2018-05-29 16:19:32,2018-05-29 17:18:09,2018-05-30 14:09:00,2018-06-01 15:48:34,2018-06-21,49.99,8.88,sao paulo,SP,moveis_sala,furniture_living_room,7875.00,sao paulo,SP,5.00,58.87,1.00,debit_card
99819,e257ae8610fb4fb68a1f459c3a4b1f51,1,a50acd33ba7a8da8e9db65094fa990a4,8581055ce74af1daba164fdbd55a40de,6d417685c0ddaca4f3926f98d0601768,delivered,2017-05-13 07:49:29,2017-05-13 08:02:19,2017-05-16 08:39:35,2017-05-19 07:29:09,2017-05-24,117.30,14.43,santo andre,SP,automotivo,auto,4105.00,guarulhos,SP,5.00,131.73,1.00,credit_card
41297,5e114d8e3840661abc3d9c4820f427b3,1,5cca3efb9521cc1d7099d610d4a12017,3d871de0142ce09b7081e2b9d1733cb1,16fa21bb47836bc052c05c338032359f,delivered,2018-04-26 18:01:20,2018-04-26 18:15:09,2018-04-30 07:37:00,2018-05-04 18:19:56,2018-05-17,58.90,13.77,sao bernardo do campo,SP,brinquedos,toys,5700.00,campo limpo paulista,SP,3.00,72.67,1.00,credit_card


---
## 13 — Save the ABT

Write the completed ABT to the `data/processed/` directory as a CSV.  
This file is the input for all downstream analysis notebooks and dashboards.

In [21]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "abt_order_items.csv"

abt.to_csv(OUTPUT_PATH, index=False)

print(f"ABT saved → {OUTPUT_PATH}")
print(f"Shape: {abt.shape[0]:,} rows × {abt.shape[1]} columns")

ABT saved → /Users/vishwashpatidar/Project Gyakriti/Gyakriti/data/processed/abt_order_items.csv
Shape: 112,650 rows × 24 columns
